# NIH Chest X-Ray Training Pipeline
This notebook trains a DenseNet-121 model for multi-label classification of 5 conditions: Pneumonia, Pleural Effusion, Atelectasis, Cardiomegaly, and Pneumothorax. It handles 'No Finding' as the baseline class.

In [ ]:
!pip install scikit-learn matplotlib
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Configuration & Setup

In [ ]:
# Configuration
DATA_DIR = './nih-chest-xrays/data'  # Update with Kaggle/Colab path
CSV_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
TRAIN_VAL_LIST = os.path.join(DATA_DIR, 'train_val_list.txt')
TEST_LIST = os.path.join(DATA_DIR, 'test_list.txt')

TARGET_CONDITIONS = ['Pneumonia', 'Effusion', 'Atelectasis', 'Cardiomegaly', 'Pneumothorax']
BATCH_SIZE = 32
LEARNING_RATE_FROZEN = 1e-4
LEARNING_RATE_UNFROZEN = 1e-5
EPOCHS = 30
PATIENCE = 5

## Data Loading, Filtering & Splitting
Here we load `Data_Entry_2017.csv` and filter it down to only our target conditions and 'No Finding'.

In [ ]:
def load_and_preprocess_data():
    print("Loading dataset labels...")
    df = pd.read_csv(CSV_PATH)
    print(f"Original dataset size: {len(df)} images")
    
    # Filter target conditions and No Finding
    def get_target_labels(labels_str):
        labels = labels_str.split('|')
        has_target = any(cond in labels for cond in TARGET_CONDITIONS)
        has_no_finding = 'No Finding' in labels
        if has_target or has_no_finding:
            return True
        return False

    # Filter dataset
    df = df[df['Finding Labels'].apply(get_target_labels)].copy()
    print(f"Filtered dataset size: {len(df)} images")
    
    # Create binary columns for target conditions
    for condition in TARGET_CONDITIONS:
        df[condition] = df['Finding Labels'].apply(lambda x: 1.0 if condition in x else 0.0)
        
    # Show class distribution
    print("\nClass Distribution after filtering:")
    for condition in TARGET_CONDITIONS:
        print(f"{condition}: {df[condition].sum():.0f}")
    no_finding_count = len(df[df['Finding Labels'] == 'No Finding'])
    print(f"No Finding: {no_finding_count}\n")
        
    # Split data using official splits to prevent patient leakage
    with open(TRAIN_VAL_LIST, 'r') as f:
        train_val_images = [line.strip() for line in f.readlines()]
    with open(TEST_LIST, 'r') as f:
        test_images = [line.strip() for line in f.readlines()]
        
    train_val_df = df[df['Image Index'].isin(train_val_images)]
    test_df = df[df['Image Index'].isin(test_images)]
    
    # Split train_val_df into train (70% total) and val (15% total)
    # Using train_test_split
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_val_df, test_size=0.176, random_state=42)
    
    return train_df, val_df, test_df

# Uncomment below to execute data loading
# train_df, val_df, test_df = load_and_preprocess_data()
# print(f'\nFinal Splits - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


## Dataset Definition & Augmentation

In [ ]:
class NIHChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.labels = self.dataframe[TARGET_CONDITIONS].values
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['Image Index']
        img_path = os.path.join(self.image_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        label = torch.FloatTensor(self.labels[idx])
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Uncomment to instantiate datasets
# train_dataset = NIHChestXrayDataset(train_df, IMAGE_DIR, transform=train_transform)
# val_dataset = NIHChestXrayDataset(val_df, IMAGE_DIR, transform=eval_transform)
# test_dataset = NIHChestXrayDataset(test_df, IMAGE_DIR, transform=eval_transform)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)


## Visualize Sample Data
Let's visualize some preprocessed images from the training set with their true labels.

In [ ]:
def show_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(1, num_samples, figsize=(16, 4))
    for i in range(num_samples):
        image, label = dataset[i]
        
        # Un-normalize for displaying
        image = image.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = std * image + mean
        image = np.clip(image, 0, 1)
        
        ax = axes[i]
        ax.imshow(image)
        
        # Get text for labels
        active_labels = [TARGET_CONDITIONS[j] for j in range(5) if label[j] == 1.0]
        title = ', '.join(active_labels) if active_labels else 'No Finding'
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

# Uncomment to show samples
# show_samples(train_dataset, 4)

## Model Building

In [ ]:
def build_model():
    model = torchvision.models.densenet121(pretrained=True)
    
    for param in model.parameters():
        param.requires_grad = False
        
    for param in model.features.denseblock3.parameters():
        param.requires_grad = True
    for param in model.features.denseblock4.parameters():
        param.requires_grad = True
    for param in model.features.norm5.parameters():
        param.requires_grad = True
        
    model.classifier = nn.Linear(1024, 5)
    return model.to(device)

model = build_model()

## Loss Function & Optimizers

In [ ]:
criterion = nn.BCEWithLogitsLoss() 

optimizer = optim.Adam([
    {'params': model.features.denseblock3.parameters(), 'lr': LEARNING_RATE_UNFROZEN},
    {'params': model.features.denseblock4.parameters(), 'lr': LEARNING_RATE_UNFROZEN},
    {'params': model.features.norm5.parameters(), 'lr': LEARNING_RATE_UNFROZEN},
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE_FROZEN}
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.1, verbose=True)

## Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    try:
        auc_scores = roc_auc_score(all_labels, all_preds, average=None)
        mean_auc = np.mean(auc_scores)
    except ValueError:
        auc_scores = [0.0] * 5
        mean_auc = 0.0
        
    return mean_auc, auc_scores

In [ ]:
best_auc = 0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    # Uncomment to train
    # train_loss = train_epoch(model, train_loader, criterion, optimizer)
    # val_auc, val_auc_scores = evaluate(model, val_loader)
    
    # Dummy values for demonstration
    train_loss = 0.0
    val_auc = 0.0
    val_auc_scores = [0.0] * 5
    
    print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - Val AUC: {val_auc:.4f}')
    
    scheduler.step(val_auc)
    
    if val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'densenet121.pt')
        print('Saved best model!')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print('Early stopping triggered!')
            break